Primeiro vamos analisar as imagens, se estão no mesmo formato e do mesmo tamanho

In [3]:
from pathlib import Path
from PIL import Image

pasta_dados = Path("../data/raw")

for classe in ["monster", "nao_monster"]:
    pasta_classe = pasta_dados / classe
    arquivos = list(pasta_classe.glob("*"))
    
    extensoes = set()
    modos_cor = set()
    tamanhos = []
    corrompidas = 0

    for arquivo in arquivos:
        extensoes.add(arquivo.suffix.lower())
        try:
            with Image.open(arquivo) as img:
                modos_cor.add(img.mode)
                tamanhos.append(img.size)  # (largura, altura)
        except Exception:
            corrompidas += 1

    print(f"=== Classe: {classe} ===")
    print(f"Total de arquivos: {len(arquivos)}")
    print(f"Extensões encontradas: {extensoes}")
    print(f"Modos de cor: {modos_cor}")
    print(f"Corrompidas: {corrompidas}")
    print(f"Exemplo de tamanhos (LxA): {tamanhos[:3]}\n")

=== Classe: monster ===
Total de arquivos: 24
Extensões encontradas: {'.png', '.jpg'}
Modos de cor: {'RGB', 'RGBA'}
Corrompidas: 0
Exemplo de tamanhos (LxA): [(1816, 4032), (1816, 4032), (1816, 4032)]

=== Classe: nao_monster ===
Total de arquivos: 32
Extensões encontradas: {'.png', '.jpg', '.avif'}
Modos de cor: {'RGB', 'RGBA'}
Corrompidas: 0
Exemplo de tamanhos (LxA): [(3000, 4000), (3000, 4000), (3024, 4032)]



Agora será realizada a padronização das imagens

In [18]:
from PIL import ImageOps

pasta_raw = Path("../data/raw")
pasta_processed = Path("../data/processed")

for classe in ["monster", "nao_monster"]:
    pasta_classe = pasta_raw / classe
    (pasta_processed / classe).mkdir(parents=True, exist_ok=True)
    arquivos = list(pasta_classe.glob("*"))

    for arquivo in arquivos:
        with Image.open(arquivo) as img:
            img = img.convert("RGB")
            img = ImageOps.pad(img, (224, 224))
            img.save(f"{pasta_processed}/{classe}/{arquivo.stem}.jpg")

Com as imagens já processadas vamos dividir em treino e teste

In [ ]:
from sklearn.model_selection import train_test_split
import shutil

dados_processed = Path("../data/processed")

arquivos = []
rotulos = []
for arquivo in dados_processed.rglob("*.jpg"):
    arquivos.append(arquivo)
    rotulos.append(arquivo.parent.name)


random_state = 174180
X_train, X_test, y_train, y_test = train_test_split(arquivos, rotulos, test_size=0.2, random_state=random_state, stratify=rotulos)

for arquivo in X_train:
    pasta_destino = Path("../data/split") / "train" / arquivo.parent.name
    pasta_destino.mkdir(parents=True, exist_ok=True)
    shutil.copy(arquivo, pasta_destino / arquivo.name)

for arquivo in X_test:
    pasta_destino = Path("../data/split") / "test" / arquivo.parent.name
    pasta_destino.mkdir(parents=True, exist_ok=True)
    shutil.copy(arquivo, pasta_destino / arquivo.name)